## Read sorting results from running the openephys pipeline


In [ ]:
import numpy as np
from pathlib import Path
import pandas as pd
def load_results_folder(folder, raw_folder = None,):
    # we load the results in a dictionary so we don't accidentally confuse results from different sessions
    folder = Path(folder)
    timestamps = None
    if not raw_folder is None:
        timestamps = list(Path(raw_folder).rglob('*timestamps.npy'))
        if len(timestamps):
            timestamps = np.load(timestamps[0])
        else:
            timestamps = None
            print('Could not find the timestamps.npy')
    res = dict(
        # spiketimes and other
        spike_times = np.load(folder.rglob('spike_times.npy').__next__()),
        spike_clusters = np.load(folder.rglob('spike_clusters.npy').__next__()),
        spike_positions = np.load(folder.rglob('spike_positions.npy').__next__()),

        spike_templates = np.load(folder.rglob('spike_templates.npy').__next__()),
        pc_features = np.load(folder.rglob('pc_features.npy').__next__()),
        pc_feature_ind = np.load(folder.rglob('pc_feature_ind.npy').__next__()),
        spike_amplitudes = np.load(folder.rglob('amplitudes.npy').__next__()),
        # metrics for each cluster
        metrics = pd.read_csv(folder.rglob('metrics.csv').__next__()),
        ccgs = np.load(folder.rglob('ccgs.npy').__next__()),
        ccgs_bins = np.load(folder.rglob('bins.npy').__next__()),
        channel_indices = np.load(folder.rglob('channel_map.npy').__next__()),
        channel_positions = np.load(folder.rglob('channel_positions.npy').__next__()),
        mean_waveforms = np.load(folder.rglob('waveforms.npy').__next__()),
        templates = np.load(folder.rglob('templates.npy').__next__()),
        whitening_mat_inv = np.load(folder.rglob('whitening_mat_inv.npy').__next__())
    )
    if not timestamps is None:
        res['spike_times'] = timestamps[res['spike_times']]
    if  not 'cluster_id' in res['metrics'].columns:
        res['metrics'].rename(columns={'Unnamed: 0': 'cluster_id'}, inplace=True)
        res['metrics'].loc[:,'cluster_id'] = np.unique(res['spike_clusters'])
    return res

res = load_results_folder('/Users/jcouto/course_data/D9_example/kilosort4_output/',raw_folder = '/Users/jcouto/course_data/D9_example/Record Node 101/experiment1/recording3/continuous/Neuropix-PXI-128.ProbeA')

In [ ]:
res['metrics']

Load the stimulus

In [ ]:
# load the stimulus
stimlog = pd.read_csv('/Users/jcouto/course_data/D9_example/grating_df.csv')

Plot a rastermap

In [ ]:
%matplotlib ipympl
import pylab as plt

plt.figure()
idx = np.random.choice(np.arange(len(res['spike_times'])),size = np.min([len(res['spike_times']),100000]))
idx = idx[np.argsort(res['spike_amplitudes'][idx])]
plt.scatter(res['spike_times'][idx],res['spike_positions'][idx,1],1,res['spike_amplitudes'][idx],alpha = 0.4)
plt.vlines(stimlog.timestamp,0,700,lw = 0.2)
